# Account Category Distribution Check

Purpose:

- Read the company CSV.
- Calculate `Accounts_AccountCategory` distribution.
- Exclude `DORMANT` and `NO ACCOUNTS FILED`.
- Show remaining category proportions for proportional stratified sampling.
- Optionally calculate suggested counts for a target sample size, without writing a new company list.

## 1. Configuration

In [3]:
from pathlib import Path
from collections import Counter
import pandas as pd

# Change this if your CSV is stored elsewhere.
INPUT_CSV = Path(r"E:\\000硕士毕设\\公司选取\\UKcompanies_8_sectors_cleaned.csv")

ACCOUNT_CATEGORY_COL = "Accounts_AccountCategory"
EXCLUDE_CATEGORIES = {"DORMANT", "NO ACCOUNTS FILED"}

# Used only to calculate suggested category counts. No output company file will be generated.
TARGET_SAMPLE_SIZE = 100_000

CHUNKSIZE = 300_000
CSV_ENCODING_CANDIDATES = ["utf-8-sig", "utf-8", "gb18030"]

print("Input CSV:", INPUT_CSV)
print("Category column:", ACCOUNT_CATEGORY_COL)
print("Excluded categories:", sorted(EXCLUDE_CATEGORIES))
print("Target sample size for reference:", TARGET_SAMPLE_SIZE)

Input CSV: E:\000硕士毕设\公司选取\UKcompanies_8_sectors_cleaned.csv
Category column: Accounts_AccountCategory
Excluded categories: ['DORMANT', 'NO ACCOUNTS FILED']
Target sample size for reference: 100000


## 2. Read Category Column and Count Distribution

In [4]:
def find_working_encoding(path, encodings):
    last_error = None
    for enc in encodings:
        try:
            pd.read_csv(path, nrows=5, encoding=enc)
            return enc
        except Exception as exc:
            last_error = exc
    raise last_error


if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")

encoding = find_working_encoding(INPUT_CSV, CSV_ENCODING_CANDIDATES)
print("Using encoding:", encoding)

header = pd.read_csv(INPUT_CSV, nrows=0, encoding=encoding)
columns = list(header.columns)
if ACCOUNT_CATEGORY_COL not in columns:
    print("Available columns:")
    for c in columns:
        print(" -", c)
    raise KeyError(f"Column not found: {ACCOUNT_CATEGORY_COL}")

counter = Counter()
total_rows = 0
missing_rows = 0

for i, chunk in enumerate(
    pd.read_csv(
        INPUT_CSV,
        usecols=[ACCOUNT_CATEGORY_COL],
        chunksize=CHUNKSIZE,
        encoding=encoding,
        low_memory=False,
    ),
    start=1,
):
    s = chunk[ACCOUNT_CATEGORY_COL].astype("string").str.strip()
    missing_rows += int(s.isna().sum() + (s == "").sum())
    values = s.dropna()
    values = values[values != ""]
    counter.update(values.tolist())
    total_rows += len(chunk)
    print(f"Processed chunk {i:,}; rows scanned: {total_rows:,}")

print("Done.")
print("Total rows:", f"{total_rows:,}")
print("Missing / blank category rows:", f"{missing_rows:,}")

Using encoding: utf-8-sig
Processed chunk 1; rows scanned: 300,000
Processed chunk 2; rows scanned: 600,000
Processed chunk 3; rows scanned: 900,000
Processed chunk 4; rows scanned: 1,200,000
Processed chunk 5; rows scanned: 1,500,000
Processed chunk 6; rows scanned: 1,800,000
Processed chunk 7; rows scanned: 2,100,000
Processed chunk 8; rows scanned: 2,400,000
Processed chunk 9; rows scanned: 2,700,000
Processed chunk 10; rows scanned: 3,000,000
Processed chunk 11; rows scanned: 3,300,000
Processed chunk 12; rows scanned: 3,415,689
Done.
Total rows: 3,415,689
Missing / blank category rows: 0


## 3. Full Category Distribution

In [5]:
full_distribution = pd.DataFrame(
    [{ACCOUNT_CATEGORY_COL: category, "count": count} for category, count in counter.most_common()]
)
full_distribution["percent_of_total"] = full_distribution["count"] / total_rows
full_distribution["exclude_from_sampling"] = full_distribution[ACCOUNT_CATEGORY_COL].isin(EXCLUDE_CATEGORIES)

full_distribution

,Accounts_AccountCategory,count,percent_of_total,exclude_from_sampling
0,MICRO ENTITY,1177923,0.344857,False
1,NO ACCOUNTS FILED,824317,0.241333,True
2,TOTAL EXEMPTION FULL,801000,0.234506,False
3,DORMANT,397028,0.116237,True
4,UNAUDITED ABRIDGED,99992,0.029274,False
5,FULL,38953,0.011404,False
6,SMALL,37560,0.010996,False
7,AUDIT EXEMPTION SUBSIDIARY,18706,0.005476,False
8,GROUP,13214,0.003869,False
9,MEDIUM,3766,0.001103,False


## 4. Distribution After Excluding Dormant / No Accounts Filed

In [6]:
eligible_distribution = full_distribution[~full_distribution[ACCOUNT_CATEGORY_COL].isin(EXCLUDE_CATEGORIES)].copy()
eligible_total = int(eligible_distribution["count"].sum())
excluded_total = int(full_distribution[full_distribution[ACCOUNT_CATEGORY_COL].isin(EXCLUDE_CATEGORIES)]["count"].sum())

eligible_distribution["percent_of_eligible"] = eligible_distribution["count"] / eligible_total
eligible_distribution["suggested_count_for_target_sample"] = (
    eligible_distribution["percent_of_eligible"] * TARGET_SAMPLE_SIZE
).round().astype(int)

summary = pd.DataFrame([
    {"metric": "total_rows", "value": total_rows},
    {"metric": "missing_or_blank_category_rows", "value": missing_rows},
    {"metric": "excluded_rows_DORMANT_NO_ACCOUNTS_FILED", "value": excluded_total},
    {"metric": "eligible_rows_after_exclusion", "value": eligible_total},
    {"metric": "excluded_percent_of_total", "value": excluded_total / total_rows if total_rows else None},
    {"metric": "eligible_percent_of_total", "value": eligible_total / total_rows if total_rows else None},
    {"metric": "target_sample_size_reference", "value": TARGET_SAMPLE_SIZE},
    {"metric": "suggested_counts_sum", "value": int(eligible_distribution["suggested_count_for_target_sample"].sum())},
])

summary

,metric,value
0,total_rows,3.415689e+06
1,missing_or_blank_category_rows,0.000000e+00
2,excluded_rows_DORMANT_NO_ACCOUNTS_FILED,1.221345e+06
3,eligible_rows_after_exclusion,2.194344e+06
4,excluded_percent_of_total,3.575691e-01
5,eligible_percent_of_total,6.424309e-01
6,target_sample_size_reference,1.000000e+05
7,suggested_counts_sum,1.000000e+05


In [7]:
eligible_distribution

,Accounts_AccountCategory,count,percent_of_total,exclude_from_sampling,percent_of_eligible,suggested_count_for_target_sample
0,MICRO ENTITY,1177923,0.344857,False,0.536800,53680
2,TOTAL EXEMPTION FULL,801000,0.234506,False,0.365029,36503
4,UNAUDITED ABRIDGED,99992,0.029274,False,0.045568,4557
5,FULL,38953,0.011404,False,0.017752,1775
6,SMALL,37560,0.010996,False,0.017117,1712
7,AUDIT EXEMPTION SUBSIDIARY,18706,0.005476,False,0.008525,852
8,GROUP,13214,0.003869,False,0.006022,602
9,MEDIUM,3766,0.001103,False,0.001716,172
10,TOTAL EXEMPTION SMALL,2243,0.000657,False,0.001022,102
11,AUDITED ABRIDGED,745,0.000218,False,0.000340,34


## 5. Plain-English Interpretation Helper

In [8]:
print("Sampling rule:")
print("1. Exclude DORMANT and NO ACCOUNTS FILED.")
print("2. Among remaining companies, sample proportionally by Accounts_AccountCategory.")
print("3. For a target sample size of", f"{TARGET_SAMPLE_SIZE:,}", "companies, use suggested_count_for_target_sample as the category quota reference.")
print()
print("Top eligible categories by count:")
display(eligible_distribution.head(10))

Sampling rule:
1. Exclude DORMANT and NO ACCOUNTS FILED.
2. Among remaining companies, sample proportionally by Accounts_AccountCategory.
3. For a target sample size of 100,000 companies, use suggested_count_for_target_sample as the category quota reference.

Top eligible categories by count:


,Accounts_AccountCategory,count,percent_of_total,exclude_from_sampling,percent_of_eligible,suggested_count_for_target_sample
0,MICRO ENTITY,1177923,0.344857,False,0.536800,53680
2,TOTAL EXEMPTION FULL,801000,0.234506,False,0.365029,36503
4,UNAUDITED ABRIDGED,99992,0.029274,False,0.045568,4557
5,FULL,38953,0.011404,False,0.017752,1775
6,SMALL,37560,0.010996,False,0.017117,1712
7,AUDIT EXEMPTION SUBSIDIARY,18706,0.005476,False,0.008525,852
8,GROUP,13214,0.003869,False,0.006022,602
9,MEDIUM,3766,0.001103,False,0.001716,172
10,TOTAL EXEMPTION SMALL,2243,0.000657,False,0.001022,102
11,AUDITED ABRIDGED,745,0.000218,False,0.000340,34
